# 2. Вопросы к изображению

Дообучим Qwen3.5-0.8B отвечать на вопросы о птицах.

In [ ]:
from pathlib import Path
import os, sys
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
ROOT = Path.cwd()
if ROOT.name in {"notebooks", "solutions"}:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = Path(os.environ.get("BIRD_DATA", str(ROOT / "data/cub8")))
assert (DATA / "manifest.jsonl").exists(), "Run scripts/prepare_data.py first"
import torch
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(4)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

## TODO 1 — LoRALinear
$y=W_0x+b+(\alpha/r)BAx$, где $A\in R^{r\times d_{in}}$, $B\in R^{d_{out}\times r}$.
Заморозьте base, инициализируйте A случайно, B нулями. Напишите forward и merged().

До запуска предскажите: какой градиент на первом шаге равен нулю? Почему обе матрицы
нельзя занулить? Сколько параметров при d_in=d_out=1024 и r=8?

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=4, alpha=4):
        super().__init__()
        raise NotImplementedError("TODO: frozen base, A, B, scaling")
    def forward(self, x):
        raise NotImplementedError("TODO: low-rank update")
    def merged(self):
        raise NotImplementedError("TODO: return equivalent nn.Linear")

In [ ]:
base = nn.Linear(5, 3)
adapter = LoRALinear(base, rank=2)
x = torch.randn(4, 5)
torch.testing.assert_close(adapter(x), base(x))
adapter(x).square().sum().backward()
assert adapter.A.grad.abs().sum() == 0
assert adapter.B.grad.abs().sum() > 0
assert base.weight.grad is None
with torch.no_grad():
    adapter.B.add_(.1)
torch.testing.assert_close(adapter(x), adapter.merged()(x))

In [ ]:
from birdlab.data import make_qa
from birdlab.vlm import load_model, QACollator, evaluate, messages
train_qa = make_qa(DATA, "train", balance=True)
val_qa = make_qa(DATA, "val", heldout_wording=True)
print(len(train_qa), len(val_qa), train_qa[0])
from transformers import set_seed
set_seed(42)
model, processor = load_model(target_mode=os.environ.get("BIRD_TARGETS", "all-linear"))

## Processor и токенизация
Рассмотрите IDs, токены и обратное декодирование. Совпадает ли число токенов с числом слов?
Посмотрите `pixel_values` и `image_grid_thw`. Почему картинку не обрабатывает tokenizer?

In [ ]:
for text in ["yes", "no", "house sparrow", "домовый воробей"]:
    ids = processor.tokenizer.encode(text, add_special_tokens=False)
    print(text, ids, processor.tokenizer.convert_ids_to_tokens(ids))
example = train_qa[0]
print(processor.apply_chat_template(messages(example, True), tokenize=False, enable_thinking=False))

## TODO 2 — маска loss
$L=-\sum_t m_t\log p(y_t|I,q,y_{<t})/\sum_t m_t$.
Верните копию input_ids; prompt и padding замените на -100. EOS ответа сохраняется.
Используйте attention_mask, а не равенство pad_token_id: pad и EOS могут совпадать.

In [ ]:
def answer_labels(input_ids, attention_mask, prompt_lengths):
    raise NotImplementedError("TODO: mask prompt and padding, preserve answer/EOS")

In [ ]:
ids = torch.tensor([[1, 2, 7, 9, 9], [1, 2, 3, 8, 9]])
attention = torch.tensor([[1, 1, 1, 1, 0], [1, 1, 1, 1, 1]])
assert answer_labels(ids, attention, [2, 3]).tolist() == [[-100,-100,7,9,-100],[-100,-100,-100,8,9]]
collator = QACollator(processor, label_function=answer_labels)
batch = collator(train_qa[:2])
for key, value in batch.items():
    print(key, tuple(value.shape))
for labels in batch["labels"]:
    print("Loss on:", processor.decode(labels[labels != -100]))
loss = model(**{k:v.to(DEVICE) for k,v in batch.items()}).loss
loss.backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for n,p in model.named_parameters() if "lora_B" in n)
model.zero_grad(set_to_none=True)
del batch, loss

## До и после LoRA

Сравните ответы до обучения и после 50 шагов LoRA.
Обучаются адаптеры языковой части; остальные веса заморожены.

In [ ]:
import random
random.Random(42).shuffle(val_qa)
evaluation = val_qa[:int(os.environ.get("BIRD_EVAL_SIZE", "24"))]
before = evaluate(model, processor, evaluation)
print(before["tasks"])
from transformers import Trainer, TrainingArguments
RUN = ROOT / "runs/notebook_vlm"
args = TrainingArguments(output_dir=str(RUN), max_steps=int(os.environ.get("BIRD_STEPS", "50")), learning_rate=2e-4,
    per_device_train_batch_size=4, gradient_accumulation_steps=2,
    remove_unused_columns=False, report_to="none", save_strategy="no", logging_steps=5,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=False)
model.config.use_cache = False
trainer = Trainer(model=model, args=args, train_dataset=train_qa, data_collator=collator)
trainer.train()
model.config.use_cache = True
after = evaluate(model, processor, evaluation)
print(after["tasks"])
model.save_pretrained(RUN / "adapter")
processor.save_pretrained(RUN / "adapter")

## Эксперименты

1. Сравните с ответом по большинству в train.
2. Перемешайте изображения и повторите оценку.
3. Сравните r=2 и r=8 при одинаковом числе шагов.
4. Почему уменьшение loss не гарантирует улучшения генерации?

In [ ]:
from birdlab.vlm import predict, MODEL_ID
from transformers import Qwen3_5ForConditionalGeneration
from peft import PeftModel
# Save a reference before releasing GPU memory; reload exactly the saved adapter.
reference = predict(model, processor, evaluation[0])
dtype = next(model.parameters()).dtype
del trainer, model
import gc
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
base = Qwen3_5ForConditionalGeneration.from_pretrained(MODEL_ID, dtype=dtype, attn_implementation="sdpa").to(DEVICE)
restored = PeftModel.from_pretrained(base, RUN / "adapter")
assert predict(restored, processor, evaluation[0]) == reference
print("Adapter reload OK:", reference)